In [46]:
# Imports
import ollama
import json
import random
from pathlib import Path
from langchain_huggingface import HuggingFaceEmbeddings
import numpy as np
import random
import time
import re
import os
import sys
import chromadb
random.seed(42)

In [47]:
os.chdir("C:/Users/filip/Desktop/Thesis/project/src")
sys.path.insert(0, "src")
os.getcwd()

'C:\\Users\\filip\\Desktop\\Thesis\\project\\src'

In [48]:

# env variables
from config import *
embed_m3 = HuggingFaceEmbeddings(model_name="BAAI/bge-m3")
embed_bge = HuggingFaceEmbeddings(model_name="BAAI/bge-base-en-v1.5")
#Chunk folders
recursive_1000 = DATA_DIR / "chunks" / "recursive_1000"
rsc = DATA_DIR / "chunks" / "rsc"
#semantic_p95 = DATA_DIR / "chunks" / "semantic_p95"

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 6825.30it/s]
BertModel LOAD REPORT from: BAAI/bge-base-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


## ζ_inf (Information Preservation)
- Randomly sample 3-sentence segments from the ORIGINAL document (not chunks)
- For each segment, have an LLM generate 1 true statement + 3 plausible false ones
- Use the true statement to retrieve relevant chunks from the vector store
- Have an LLM pick which of the 4 statements is true, using only the retrieved chunks
- If it picks correctly, the information was preserved. Binary 1/0 scoring
- Average across all samples = ζ_inf

In [49]:
def clean_sentences(doc_text):
    """Split document text into clean prose sentences, filtering junk."""
    raw_sentences = re.split(r'(?<=[.!?])\s+', doc_text)
    sentences = []
    for s in raw_sentences:
        s = s.strip()
        if len(s) < 40:
            continue
        if not re.match(r'[A-Z]', s):
            continue
        if s.count(' ') < 5:
            continue
        if '\n' in s:
            continue
        if re.search(r'\d+\.\d+\s*[+±]\s*\d+', s):  # measurements
            continue
        if re.search(r'(.)\1{4,}', s):  # repeated characters (OCR garbage)
            continue
        sentences.append(s)
    return sentences

In [50]:
def sample_segments(sentences, num_segments=5, segment_length=3):
    """Pick random consecutive 3-sentence segments from a list of sentences."""
    if len(sentences) < segment_length:
        return []
    
    max_start = len(sentences) - segment_length
    starts = random.sample(range(max_start + 1), min(num_segments, max_start + 1))
    
    segments = [" ".join(sentences[start:start + segment_length]) for start in starts]
    return segments

In [51]:
def generate_quadruple(segment, model="granite3.2:8b"):
    """Generate 1 true + 3 false statements from a document segment."""
    prompt = (
        "Read the following segment from a scientific paper. "
        "Write exactly 1 true factual statement based on the segment, "
        "and 3 plausible but false statements that sound like they could be from the same paper. "
        "The false statements should be subtle — wrong numbers, swapped species names, or inverted relationships. "
        "Return ONLY a JSON object with keys 'true' and 'false', where 'true' is a string "
        "and 'false' is a list of 3 strings. No explanation, no markdown.\n\n"
        f"Segment: {segment}"
    )
    response = ollama.chat(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        options={"temperature": 0.7},
    )
    raw = response["message"]["content"].strip()
    
    if raw.startswith("```"): #sometimes it returns backticks
        raw = raw.split("\n", 1)[1] if "\n" in raw else raw[3:]
        if "```" in raw:
            raw = raw.rsplit("```", 1)[0]
    raw = raw.strip()
    
    result = json.loads(raw)
    if "true" not in result or "false" not in result:
        raise ValueError(f"Missing keys in response: {result.keys()}")
    if len(result["false"]) != 3:
        raise ValueError(f"Expected 3 false statements, got {len(result['false'])}")
    
    return result

In [52]:
def judge_quadruple(quadruple, retrieved_chunks, model="gemma3:1b"):
    """Given a quadruple and retrieved chunks, ask LLM which statement is true."""
    # Shuffle so the true statement isn't always first
    statements = [quadruple["true"]] + quadruple["false"]
    random.shuffle(statements)
    true_index = statements.index(quadruple["true"])
    
    context = "\n\n".join(retrieved_chunks)
    numbered = "\n".join(f"{i+1}. {s}" for i, s in enumerate(statements))
    
    prompt = (
        "Based ONLY on the following context, which of the 4 statements is true? "
        "Reply with ONLY the number (1, 2, 3, or 4). Nothing else.\n\n"
        f"Context:\n{context}\n\n"
        f"Statements:\n{numbered}"
    )
    response = ollama.chat(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        options={"temperature": 0.0},  # deterministic for judging
    )
    raw = response["message"]["content"].strip()
    
    # Extract the number from response
    match = re.search(r'[1-4]', raw)
    if not match:
        return False
    
    predicted = int(match.group()) - 1  # 0-indexed
    return predicted == true_index

In [53]:
def ζ_inf_document(doc_text, doc_filename, collection, embed_model, 
                    num_segments=2, model="granite3.2:8b"): #reduced from 5 to save on computation
    """Compute information preservation for one document against one collection."""
    sentences = clean_sentences(doc_text)
    segments = sample_segments(sentences, num_segments=num_segments)
    
    if not segments:
        return {"score": np.nan, "n_segments": 0, "details": []}
    
    correct = 0
    details = []
    
    for seg in segments:
        try:
            # Generate quadruple
            quad = generate_quadruple(seg, model=model)
            
            # Retrieve chunks using the true statement as query
            query_embedding = embed_model.embed_query(quad["true"])
            results = collection.query(
                query_embeddings=[query_embedding],
                n_results=3,
                where={"filename": doc_filename}
            )
            
            retrieved = results["documents"][0] if results["documents"][0] else []
            if not retrieved:
                details.append({"segment": seg[:100], "result": "no_chunks_retrieved"})
                continue
            
            # Judge
            is_correct = judge_quadruple(quad, retrieved, model=model)
            correct += int(is_correct)
            details.append({"segment": seg[:100], "result": is_correct})
            
        except (json.JSONDecodeError, ValueError, KeyError) as e:
            details.append({"segment": seg[:100], "result": f"error: {e}"})
            continue
    
    scored = [d for d in details if isinstance(d["result"], bool)]
    score = correct / len(scored) if scored else 0.0
    
    return {"score": score, "n_segments": len(scored), "details": details}

In [54]:
ζ_inf_path = PROJECT_ROOT / "outputs" / "HOPE" / "hope_inf_m3_embedding_results.json"

if ζ_inf_path.exists():
    with open(ζ_inf_path, encoding="utf-8") as f:
        ζ_inf_results = json.load(f)
    print(f"Resuming — {len(ζ_inf_results)} entries done")
else:
    ζ_inf_results = {}

chroma_client = chromadb.PersistentClient(path=str(DATA_DIR / "chromadb"))
collections = {
    "recursive_1000_m3": chroma_client.get_collection(name="recursive_1000_m3"),
    "rsc_m3": chroma_client.get_collection(name="rsc_m3"),
    "semantic_p95": chroma_client.get_collection(name="semantic_p95"),
    "recursive_1000_bge": chroma_client.get_collection(name="recursive_100"),
    "rsc_bge": chroma_client.get_collection(name="rsc"),
}

# Reuse same 30 docs from ζ_con
with open(PROJECT_ROOT / "outputs" / "HOPE" / "hope_con_results.json", encoding="utf-8") as f:
    ζ_con_raw = json.load(f)
sampled_docs = sorted({key.split("/")[1] for key in ζ_con_raw.keys()})

processed_dir = DATA_DIR / "processed"
total = len(sampled_docs) * len(collections)
done = sum(1 for key in ζ_inf_results if any(key.startswith(s + "/") for s in collections))

In [55]:
#setup
embed_for_collection = {
    "recursive_1000_m3": embed_m3,
    "rsc_m3": embed_m3,
    "recursive_1000_bge": embed_bge,
    "rsc_bge": embed_bge,
}

In [56]:
#loading quaruples
quads_path = PROJECT_ROOT / "outputs" / "HOPE" / "quadruples.json"

if quads_path.exists():
    with open(quads_path, encoding="utf-8") as f:
        all_quads = json.load(f)
else:
    all_quads = {}

def get_quads(doc_name, doc_text):
    """Return cached quadruples, or generate and save new ones."""
    if doc_name in all_quads:
        return all_quads[doc_name]

    sentences = clean_sentences(doc_text)
    segments = sample_segments(sentences, num_segments=2)
    quads = []
    for seg in segments:
        try:
            quads.append(generate_quadruple(seg))
        except (json.JSONDecodeError, ValueError) as e:
            print(f"  Quadruple failed: {e}")

    all_quads[doc_name] = quads
    with open(quads_path, "w", encoding="utf-8") as f:
        json.dump(all_quads, f, indent=2, ensure_ascii=False)

    return quads

In [57]:
#SINGLE COLLECITON EVALUATION
def evaluate_collection(quads, doc_filename, strat_name, collection):
   # Runs quads against one collection, return score and count. ~~~
    embed_model = embed_for_collection[strat_name]
    correct = 0
    scored = 0

    for quad in quads:
        query_embedding = embed_model.embed_query(quad["true"])
        results = collection.query(
            query_embeddings=[query_embedding],
            n_results=3,
            where={"filename": doc_filename}
        )
        retrieved = results["documents"][0] if results["documents"][0] else []
        if not retrieved:
            continue
        if judge_quadruple(quad, retrieved):
            correct += 1
        scored += 1

    score = correct / scored if scored else 0.0
    return score, scored

In [58]:
#LOOP to iterate over all
total = len(sampled_docs) * len(collections)
done = len(ζ_inf_results)

for doc_name in sampled_docs:
    processed_path = processed_dir / doc_name
    if not processed_path.exists():
        print(f"  Skipping {doc_name} — no processed file")
        done += len(collections)
        continue

    if all(f"{s}/{doc_name}" in ζ_inf_results for s in collections):
        done += len(collections)
        continue

    with open(processed_path, encoding="utf-8") as f:
        doc = json.load(f)

    quads = get_quads(doc_name, doc["text"])
    if not quads:
        print(f"  No valid quadruples for {doc_name}, skipping")
        done += len(collections)
        continue

    for strat_name, collection in collections.items():
        key = f"{strat_name}/{doc_name}"
        if key in ζ_inf_results:
            done += 1
            continue

        start = time.time()
        score, scored = evaluate_collection(quads, doc["filename"], strat_name, collection)
        elapsed = time.time() - start

        ζ_inf_results[key] = {
            "score": score,
            "n_segments": scored,
            "elapsed_s": round(elapsed, 1)
        }
        with open(ζ_inf_path, "w", encoding="utf-8") as f:
            json.dump(ζ_inf_results, f, indent=2, ensure_ascii=False)
        done += 1
        print(f"[{done}/{total}] {key} — ζ_inf={score:.2f} ({scored} segments, {elapsed:.0f}s)")

print("\nDone! Results at:", ζ_inf_path)

ResponseError: llama runner process has terminated: error loading model: unable to allocate CPU buffer (status code: 500)